# CPU：全量 EEG / MEG 缓存预热
选择纯 CPU 运行时，依次执行。源码只从 GitHub 获取，不嵌入源码包。
原始数据保留在 Drive；本流程调用同一套 `Experiment.data.build()`，不创建神经网络或 GPU Trainer。
首次全量预处理可能很久；重跑会复用已完成的 timeline 和信号缓存。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
REPO_URL = "https://github.com/wdongdong0708-jpg/brain2qwertyv1Test.git"
CODE = Path("/content/code")
WORKSPACE = Path("/content/drive/MyDrive/Brain2Qwerty_Workspace")
if (CODE / ".git").exists():
    subprocess.run(["git", "-C", str(CODE), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(CODE)], check=True)
os.chdir(CODE)
for folder in ("data", "cache_persisted", "archives", "runs"):
    (WORKSPACE / folder).mkdir(parents=True, exist_ok=True)
raw = WORKSPACE / "data/SpanishBCBL_Full"
if not raw.is_dir():
    raise FileNotFoundError(f"Put the already downloaded full SpanishBCBL repository at {raw}")
for name in ("Pinet2024Eeg", "Pinet2024Meg"):
    alias = WORKSPACE / "data" / name
    if not alias.exists():
        try:
            alias.symlink_to(raw, target_is_directory=True)
        except OSError:
            print(f"Drive does not support this symlink; the reader will resolve {raw} directly")
os.environ["BRAIN2QWERTY_STUDIES"] = str(WORKSPACE / "data")
os.environ["BRAIN2QWERTY_RESULTS"] = str(WORKSPACE / "runs")
subprocess.run(["git", "log", "-1", "--oneline"], check=True)


In [ ]:
subprocess.run([sys.executable, "-m", "scripts.colab_setup"], check=True)
PYTHON = "/content/b2q-venv/bin/python"


In [ ]:
os.environ["BRAIN2QWERTY_CACHE"] = str(WORKSPACE / "cache_persisted")
MODALITIES = ["eeg", "meg"]
CPU_THREADS = 2
for modality in MODALITIES:
    subprocess.run([PYTHON, "-u", "-m", "brain2qwerty_v1", "cache",
                    "--modality", modality, "--cpu-threads", str(CPU_THREADS)], check=True)


可选：所有缓存完成后生成一个未压缩 tar。后续 GPU 实例顺序读取 tar 并解压到本地磁盘。打包时不要同时启动写入这些缓存的任务。


In [ ]:
subprocess.run([PYTHON, "-m", "scripts.cache_archive", "pack",
                "--cache-root", os.environ["BRAIN2QWERTY_CACHE"],
                "--archive", str(WORKSPACE / "archives/features.tar")], check=True)
